# RX Patient Features

This notebook generates patient-level features for **Rx (prescription) claims** using the Generic Patient Feature Engineering methodology.

## Features Generated

1. **First Occurrence** -- Days before anchor when each drug first appeared in a patient's journey (full lookback)
2. **Recency** -- How recently/frequently events occurred (12M lookback, 6 sub-features)
3. **Frequency** -- Monthly claim counts, rolling lags, change in frequency, unique patient counts
4. **Hospitalization** -- NOT applicable for RX (DX/PX only)

## Source Mapping (RX-specific)

| Canonical | RX Raw |
|---|---|
| `EVENT_NAME` | `PRODUCT_GROUP` |
| `EVENT_TYPE` | `rx` |
| `EVENT_DATE` | `SVC_DT` |

**Column naming for Freq Sub-D:** `NUM_OF_PATIENTS_THE_HCP_PRESCRIBED_WITH_{EVENT}_IN_LAST_{N}_DAYS`

In [0]:
# ============================================================================
# CONFIG PARAMETERS (from 00_config)
# ============================================================================

# Cohort parameters
cohort_start_date = "2026-06-01"       # Single cohort: June 2026
cohort_end_date = "2026-06-01"          # Same as start = generate 1 cohort only
months_increment = 2                    # Months between cohorts
lookback_duration = 365                 # Days of history per cohort
prediction_duration = 90                 # Days after anchor for label
cohort_day_offset = 1                    # Day of month that starts each custom-month (1 = calendar month)

# Sampling parameters
sample_hcp_count = 10000                 # Randomly sample N HCPs (0 = use all HCPs)

# Feature-specific parameters
event_prevalence = 0.00                  # Keep all events (0.00 = no filter)
relevant_rx_events = ['STEGLATRO', 'INVOKANA', 'KERENDIA']  # Force-keep these drugs
batch_size = 6                           # Cohorts per batch

# Recency time windows (days)
time_windows_recency = [30, 60, 90, 180, 270, 360]

# Frequency lag windows (months)
lag_windows_freq = [1, 2, 3, 6, 12]

# Frequency change windows (months, excludes 12M which is the baseline)
change_windows_freq = [1, 2, 3, 6]

# Frequency patient count windows (days)
patient_count_windows_freq = [30, 60, 90, 180, 360]

# File paths (update these to your actual paths)
rx_file_path = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/query_output/rx_input_claims_total_events_and_first_occ_features_input"
output_base_path = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/"
anchor_dates_path = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_volume/hcp_level_model/ops_pipeline/notebook_output/hcp_model_inference_anchor_dates.csv"

print("Config parameters loaded.")

In [0]:
# ============================================================================
# IMPORTS AND STANDARDIZATION FUNCTION
# ============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel
import pandas as pd
from datetime import datetime, timedelta

# -- Column mapping from raw source variants to canonical schema (Section 1.3) --
COLUMN_MAP = {
    "PROVIDER_ID": ["PROVIDER_ID", "HCP_ID", "PHYSICIAN_ID", "PROV_ID", "BHID"],
    "PATIENT_ID":  ["PATIENT_ID", "MEMBER_ID", "ENROLLEE_ID", "SUBSCRIBER_ID"],
    "EVENT_NAME":  ["PRODUCT_GROUP", "DIAGNOSIS_DESCRIPTION",
                    "PROCEDURE_DESCRIPTION", "TEST_TYPE"],
    "EVENT_DATE":  ["SVC_DT", "SERVICE_DATE", "CLAIM_DATE", "FILL_DT"],
    "CLAIM_ID":    ["CLAIM_ID", "CLAIM_NBR", "CLAIM_NUMBER"],
}

def standardize(df, event_type):
    """Standardize raw claims to canonical schema.
    
    Renames source-specific columns to canonical names, assigns EVENT_TYPE,
    and prunes to only the columns needed for feature computation.
    """
    for canonical, variants in COLUMN_MAP.items():
        for v in variants:
            if v in df.columns and canonical not in df.columns:
                df = df.withColumnRenamed(v, canonical)
                break
    df = df.withColumn("EVENT_TYPE", F.lit(event_type))
    keep = [c for c in
            ["PROVIDER_ID", "PATIENT_ID", "EVENT_NAME", "EVENT_DATE",
             "EVENT_TYPE", "CLAIM_ID", "DATA_SOURCE"]
            if c in df.columns]
    return df.select(*keep)

def sanitize_column_name(name):
    """Sanitize column names per skill rules: , removed, + -> and, / -> or, - -> _, spaces -> _, uppercase."""
    import re as _re
    name = _re.sub(r'[^A-Za-z0-9_]', '_', name.replace("+", "AND").replace("/", "_OR_"))
    name = _re.sub(r'_+', '_', name).strip('_')
    return name.upper()

print("Standardization functions loaded.")

In [0]:
# ============================================================================
# LOAD AND STANDARDIZE RX CLAIMS
# ============================================================================

# --- Load from pre-processed Delta table and standardize to canonical schema ---
raw_df = spark.read.format("delta").load(rx_file_path)
rx_claims = standardize(raw_df, "rx")

# Ensure EVENT_DATE is date type
rx_claims = rx_claims.withColumn("EVENT_DATE", F.col("EVENT_DATE").cast("date"))

# Sample 10K HCPs randomly for testing (includes both old and new claims)
if sample_hcp_count > 0:
    sampled_hcps = (rx_claims
        .select("PROVIDER_ID")
        .distinct()
        .orderBy(F.rand())
        .limit(sample_hcp_count)
    )
    rx_claims = rx_claims.join(sampled_hcps, "PROVIDER_ID", "inner")
    print(f"Sampled {sample_hcp_count} HCPs randomly")

print(f"RX claims loaded: {rx_claims.count()} rows")
display(rx_claims.limit(10))

In [0]:
# ============================================================================
# COHORT ANCHOR DATES GENERATION (Section 2)
# ============================================================================

def generate_cohort_anchor_dates(cohort_start, cohort_end, months_incr, lookback_dur, cohort_offset=1):
    """Generate cohort anchor dates and derived lookback boundaries.
    
    Per Section 2:
    - LOOKBACK_END_DATE = last day of the month, 2 months before the anchor
    - LOOKBACK_START_DATE = LOOKBACK_END_DATE - lookback_duration days
    - HCP_COHORT_ID = {PROVIDER_ID}_{COHORT}
    - SVC_DT_M = date_trunc('MM', EVENT_DATE)
    - EVENT_TIME = datediff(LOOKBACK_END_DATE, EVENT_DATE)
    
    When cohort_day_offset != 1, custom-month boundaries apply (Section 10).
    """
    anchors = []
    current = pd.Timestamp(cohort_start)
    end = pd.Timestamp(cohort_end)
    
    cohort_idx = 1
    while current <= end:
        cohort_label = f"I{cohort_idx:02d}"
        
        # LOOKBACK_END_DATE: last day of the month, 2 months before anchor
        if cohort_offset == 1:
            # Default: calendar month boundaries
            lookback_end = (current - pd.DateOffset(months=2)).to_period('M').end_time.normalize()
        else:
            # Custom month boundaries (Section 10.1)
            month_start = current.to_period('M').start_time
            if current.day >= cohort_offset:
                custom_month_start = month_start + pd.DateOffset(days=cohort_offset - 1)
            else:
                custom_month_start = (month_start - pd.DateOffset(months=1)) + pd.DateOffset(days=cohort_offset - 1)
            lookback_end = (custom_month_start - pd.DateOffset(months=1)) - pd.DateOffset(days=1)
        
        lookback_start = lookback_end - pd.Timedelta(days=lookback_dur)
        
        anchors.append({
            "COHORT": cohort_label,
            "ANCHOR_DATE": current.strftime("%Y-%m-%d"),
            "LOOKBACK_END_DATE": lookback_end.strftime("%Y-%m-%d"),
            "LOOKBACK_START_DATE": lookback_start.strftime("%Y-%m-%d"),
        })
        
        current = current + pd.DateOffset(months=months_incr)
        cohort_idx += 1
    
    anchor_df = pd.DataFrame(anchors)
    return anchor_df

# Generate cohort anchor dates
cohort_anchor_pd = generate_cohort_anchor_dates(
    cohort_start_date, cohort_end_date, months_increment, lookback_duration, cohort_day_offset
)

# Convert to Spark DataFrame for use in feature computation
cohort_anchor_dates = spark.createDataFrame(cohort_anchor_pd)
cohort_anchor_dates = cohort_anchor_dates.withColumn("ANCHOR_DATE", F.col("ANCHOR_DATE").cast("date")) \
    .withColumn("LOOKBACK_END_DATE", F.col("LOOKBACK_END_DATE").cast("date")) \
    .withColumn("LOOKBACK_START_DATE", F.col("LOOKBACK_START_DATE").cast("date"))

# Save anchor dates for downstream joins (Section 2)
cohort_anchor_pd.to_csv(anchor_dates_path, index=False)

print(f"Generated {cohort_anchor_pd.shape[0]} cohorts:")
display(cohort_anchor_dates)

---
# Feature Type 1: First Occurrence (Section 3)

## What it answers
*How many days before the anchor date did this event first appear in the patient's journey?*

## How It Works
1. Collapse raw claims to one row per (PROVIDER_ID, EVENT_NAME) with earliest EVENT_DATE
2. Cross-join with broadcast cohort anchor dates
3. Compute EVENT_TIME = datediff(LOOKBACK_END_DATE, FIRST_OCCURRENCE_DATE)
4. Filter EVENT_TIME >= 0 (future leakage guard)
5. Pivot wide by EVENT_NAME, agg max(EVENT_TIME)
6. Suffix columns with `_first_occurrence`

## Business Rules
- Lookback scope: **Full history** (EVENT_TIME >= 0, no lower bound) — **Hard**
- Future leakage guard: filter(EVENT_TIME >= 0) — **Hard**
- Core aggregation: max(EVENT_TIME) per HCP_COHORT_ID per EVENT_NAME — **Hard**
- Event prevalence filter: event_prevalence threshold (0.00 = keep all) — **Tunable**
- Bypass list: relevant_rx_events force-keeps specific drugs — **Tunable**
- Batch size: cohorts per batch (batch_size=6) — **Tunable**

## Column Naming
`max(EVENT_TIME)_{EVENT_NAME}_first_occurrence`

Example: `max(EVENT_TIME)_KERENDIA_first_occurrence = 287` means first Kerendia Rx was 287 days before anchor.

In [0]:
# ============================================================================
# FIRST OCCURRENCE - Step 1: Collapse raw claims (Section 3)
# ============================================================================
# Pre-aggregate to one row per (PROVIDER_ID, EVENT_NAME) with earliest EVENT_DATE
# This collapses ~100B raw claims to millions of rows before cohort cross-join

first_occ_intermediate = (rx_claims
    .filter(F.col("PROVIDER_ID").isNotNull())
    .filter(F.col("EVENT_NAME").isNotNull())
    .groupBy("PROVIDER_ID", "EVENT_NAME")
    .agg(F.min("EVENT_DATE").alias("FIRST_OCCURRENCE_DATE"))
)

# Write intermediate to storage (Section 7.4: checkpoint intermediates)
first_occ_intermediate_path = f"{output_base_path}/rx_first_occ_intermediate"
first_occ_intermediate.write.format("parquet").mode("overwrite").save(first_occ_intermediate_path)

# Read back from storage
first_occ_intermediate = spark.read.parquet(first_occ_intermediate_path)

print(f"First occurrence intermediate: {first_occ_intermediate.count()} rows")
display(first_occ_intermediate.limit(10))

In [0]:
# ============================================================================
# FIRST OCCURRENCE - Step 2: Attach cohort dates + Step 3: Pivot wide
# ============================================================================
# Cross-join intermediate with broadcast cohort anchor dates (~100 rows)
# Compute EVENT_TIME, apply future leakage guard, pivot wide

def compute_first_occurrence(intermediate_df, cohort_anchors, event_prev=0.00, bypass_events=None, batch_sz=6):
    """Compute first occurrence features following Section 3 methodology.
    
    Args:
        intermediate_df: Pre-aggregated (PROVIDER_ID, EVENT_NAME, FIRST_OCCURRENCE_DATE)
        cohort_anchors: Cohort anchor dates DataFrame (broadcast)
        event_prev: Event prevalence threshold (0.00 = keep all)
        bypass_events: List of events to force-keep regardless of prevalence
        batch_sz: Number of cohorts per batch
    Returns:
        DataFrame with HCP_COHORT_ID and max(EVENT_TIME)_{EVENT}_first_occurrence columns
    """
    if bypass_events is None:
        bypass_events = []
    
    # Step 2: Attach cohort dates via broadcast cross-join
    with_cohort = (intermediate_df
        .crossJoin(F.broadcast(cohort_anchors))
        .withColumn("EVENT_TIME", F.datediff(F.col("LOOKBACK_END_DATE"), F.col("FIRST_OCCURRENCE_DATE")))
        .filter(F.col("EVENT_TIME") >= 0)  # Future leakage guard (Hard)
        .withColumn("HCP_COHORT_ID", F.concat_ws("_", F.col("PROVIDER_ID"), F.col("COHORT")))
    )
    
    # Event prevalence filter (Tunable)
    if event_prev > 0.00:
        # Count distinct HCPs per event
        event_counts = (with_cohort
            .groupBy("EVENT_NAME")
            .agg(F.countDistinct("PROVIDER_ID").alias("hcp_count"))
        )
        total_hcps = with_cohort.select("PROVIDER_ID").distinct().count()
        
        # Keep events above threshold OR in bypass list
        events_to_keep = (event_counts
            .filter((F.col("hcp_count") / F.lit(total_hcps) >= event_prev) | 
                    F.col("EVENT_NAME").isin(bypass_events))
            .select("EVENT_NAME")
            .collect()
        )
        keep_list = [r["EVENT_NAME"] for r in events_to_keep]
        with_cohort = with_cohort.filter(F.col("EVENT_NAME").isin(keep_list))
    else:
        # Even with prevalence=0, still apply bypass list to ensure relevant events are kept
        if bypass_events:
            # No filter needed, bypass events are already included
            pass
    
    # Step 3: Pivot wide — max(EVENT_TIME) per HCP_COHORT_ID per EVENT_NAME
    # Batch processing by cohort (Section 7.5)
    cohort_list = [row["COHORT"] for row in cohort_anchors.select("COHORT").collect()]
    
    batch_results = []
    for i in range(0, len(cohort_list), batch_sz):
        batch_cohorts = cohort_list[i:i + batch_sz]
        batch_df = with_cohort.filter(F.col("COHORT").isin(batch_cohorts))
        
        pivoted = (batch_df
            .groupBy("HCP_COHORT_ID")
            .pivot("EVENT_NAME")
            .agg(F.max("EVENT_TIME"))
        )
        batch_results.append(pivoted)
    
    # Union batches with allowMissingColumns=True
    result = batch_results[0]
    for batch in batch_results[1:]:
        result = result.unionByName(batch, allowMissingColumns=True)
    
    # Suffix non-ID columns with _first_occurrence and sanitize names
    id_cols = ["HCP_COHORT_ID"]
    rename_exprs = []
    for col_name in result.columns:
        if col_name not in id_cols:
            sanitized = sanitize_column_name(col_name)
            rename_exprs.append(F.col(f"`{col_name}`").alias(f"MAX_EVENT_TIME_{sanitized}_FIRST_OCCURRENCE"))
        else:
            rename_exprs.append(F.col(col_name))
    result = result.select(*rename_exprs)
    
    return result

# Compute first occurrence features
first_occurrence_features = compute_first_occurrence(
    intermediate_df=first_occ_intermediate,
    cohort_anchors=cohort_anchor_dates,
    event_prev=event_prevalence,
    bypass_events=relevant_rx_events,
    batch_sz=batch_size
)

# Save output (Section 7.7: partition by COHORT)
first_occ_output_path = f"{output_base_path}/rx_first_occurrence"
first_occurrence_features.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .save(first_occ_output_path)

print(f"First occurrence features: {first_occurrence_features.count()} rows, {len(first_occurrence_features.columns)} columns")
display(first_occurrence_features.limit(10))

---
# Feature Type 2: Recency (Section 4)

## What it answers
*How recently did events occur, and how concentrated was the activity in the patient's recent history?*

## Critical Distinction from First Occurrence
- **First Occurrence** uses **full history** (all events where EVENT_TIME >= 0)
- **Recency** uses **only the 12M window** (EVENT_DATE BETWEEN LOOKBACK_START AND LOOKBACK_END)

## Six Sub-Features

| # | Sub-Feature | Method | Output Columns |
|---|---|---|---|
| 1 | Most Recent Event | `min(EVENT_TIME)` pivot | `min(EVENT_TIME)_{EVENT_NAME}_time_min` |
| 2 | Total and Unique Events | `count` / `countDistinct` | `total_events`, `unique_events` |
| 3 | Counts by Time Window | filter + count per window | `total_events_time_{N}_days`, `unique_events_time_{N}_days` |
| 4 | Counts by Type | filter by EVENT_TYPE + count | `total_{type}`, `unique_{type}` |
| 5 | Counts by Type x Window | filter by type AND window | `total_events_time_{N}_days_{type}`, `unique_events_time_{N}_days_{type}` |
| 6 | Skewness | `skewness(EVENT_TIME)` per type x window | `skewness_time_{N}_days_{type}` |

## Business Rules
- Lookback scope: 12M only — **Hard**
- Re-filtering: Claims re-filtered to 12M, then re-batched — **Hard**
- Most recent event: `min(EVENT_TIME)` per HCP_COHORT_ID per EVENT_NAME — **Hard**
- Time windows: `[30, 60, 90, 180, 270, 360]` days — **Tunable**
- Event types: `rx`, `dx`, `px`, `lab` — **Hard**

In [0]:
# ============================================================================
# RECENCY - 12M Filter + Attach Cohort Dates (Section 4)
# ============================================================================
# Critical: Recency uses ONLY the 12M window, NOT full history
# Claims are re-filtered to 12M, then re-batched

# Cross-join claims with broadcast cohort anchor dates
recency_claims = (rx_claims
    .filter(F.col("PROVIDER_ID").isNotNull())
    .filter(F.col("EVENT_NAME").isNotNull())
    .crossJoin(F.broadcast(cohort_anchor_dates))
)

# 12M filter (Hard): EVENT_DATE BETWEEN LOOKBACK_START AND LOOKBACK_END
recency_claims_12m = recency_claims.filter(
    (F.col("EVENT_DATE") >= F.col("LOOKBACK_START_DATE")) &
    (F.col("EVENT_DATE") <= F.col("LOOKBACK_END_DATE"))
)

# Compute EVENT_TIME = datediff(LOOKBACK_END_DATE, EVENT_DATE)
recency_claims_12m = recency_claims_12m.withColumn(
    "EVENT_TIME", F.datediff(F.col("LOOKBACK_END_DATE"), F.col("EVENT_DATE"))
)

# Generate HCP_COHORT_ID and SVC_DT_M
recency_claims_12m = recency_claims_12m \
    .withColumn("HCP_COHORT_ID", F.concat_ws("_", F.col("PROVIDER_ID"), F.col("COHORT"))) \
    .withColumn("SVC_DT_M", F.date_trunc("MM", F.col("EVENT_DATE")))

print(f"Recency 12M claims: {recency_claims_12m.count()} rows")
display(recency_claims_12m.limit(10))

In [0]:
# ============================================================================
# RECENCY - Sub-feature 1: Most Recent Event (Section 4)
# ============================================================================
# min(EVENT_TIME) per HCP_COHORT_ID per EVENT_NAME
# Fewest days before anchor = most recent event
# Pre-aggregation pattern (Section 7.1): collapse to (PROVIDER_ID, EVENT_NAME, LATEST_EVENT_DATE)
# before cohort cross-join to avoid 100B x N_cohorts explosion

# Pre-aggregate: latest event date per (PROVIDER_ID, EVENT_NAME)
recency_latest = (rx_claims
    .filter(F.col("PROVIDER_ID").isNotNull())
    .filter(F.col("EVENT_NAME").isNotNull())
    .groupBy("PROVIDER_ID", "EVENT_NAME")
    .agg(F.max("EVENT_DATE").alias("LATEST_EVENT_DATE"))
)

# Attach cohort dates and compute EVENT_TIME
recency_sub1 = (recency_latest
    .crossJoin(F.broadcast(cohort_anchor_dates))
    .withColumn("EVENT_TIME", F.datediff(F.col("LOOKBACK_END_DATE"), F.col("LATEST_EVENT_DATE")))
    .filter(F.col("EVENT_TIME") >= 0)  # Future leakage guard
    .withColumn("HCP_COHORT_ID", F.concat_ws("_", F.col("PROVIDER_ID"), F.col("COHORT")))
)

# 12M filter: only keep events within the 12M lookback window
recency_sub1 = recency_sub1.filter(
    (F.col("LATEST_EVENT_DATE") >= F.col("LOOKBACK_START_DATE")) &
    (F.col("LATEST_EVENT_DATE") <= F.col("LOOKBACK_END_DATE"))
)

# Pivot: min(EVENT_TIME) per HCP_COHORT_ID per EVENT_NAME
# Batch by cohort (Section 7.5)
cohort_list = [row["COHORT"] for row in cohort_anchor_dates.select("COHORT").collect()]
batch_results_sub1 = []
for i in range(0, len(cohort_list), batch_size):
    batch_cohorts = cohort_list[i:i + batch_size]
    batch_df = recency_sub1.filter(F.col("COHORT").isin(batch_cohorts))
    pivoted = (batch_df
        .groupBy("HCP_COHORT_ID")
        .pivot("EVENT_NAME")
        .agg(F.min("EVENT_TIME"))
    )
    batch_results_sub1.append(pivoted)

# Union batches
recency_sub1_pivot = batch_results_sub1[0]
for batch in batch_results_sub1[1:]:
    recency_sub1_pivot = recency_sub1_pivot.unionByName(batch, allowMissingColumns=True)

# Suffix columns with _time_min
id_cols = ["HCP_COHORT_ID"]
rename_exprs = []
for col_name in recency_sub1_pivot.columns:
    if col_name not in id_cols:
        sanitized = sanitize_column_name(col_name)
        rename_exprs.append(F.col(f"`{col_name}`").alias(f"MIN_EVENT_TIME_{sanitized}_TIME_MIN"))
    else:
        rename_exprs.append(F.col(col_name))
recency_sub1_result = recency_sub1_pivot.select(*rename_exprs)

print(f"Recency Sub-1 (Most Recent Event): {recency_sub1_result.count()} rows")
display(recency_sub1_result.limit(10))

In [0]:
# ============================================================================
# RECENCY - Sub-features 2-6 (Section 4)
# ============================================================================
# Uses the 12M-filtered claims (recency_claims_12m) computed earlier

# --- Sub-feature 2: Total and Unique Events ---
# Overall volume per HCP-Cohort
recency_sub2 = (recency_claims_12m
    .groupBy("HCP_COHORT_ID")
    .agg(
        F.count("*").alias("total_events"),
        F.countDistinct("EVENT_NAME").alias("unique_events")
    )
)

print(f"Recency Sub-2 (Total & Unique Events): {recency_sub2.count()} rows")

# --- Sub-feature 3: Counts by Time Window ---
# Events in last N days: total_events_time_{N}_days, unique_events_time_{N}_days
recency_sub3_agg = recency_claims_12m.select("HCP_COHORT_ID").distinct()
for window_n in time_windows_recency:
    window_df = recency_claims_12m.filter(F.col("EVENT_TIME") <= window_n)
    window_counts = (window_df
        .groupBy("HCP_COHORT_ID")
        .agg(
            F.count("*").alias(f"total_events_time_{window_n}_days"),
            F.countDistinct("EVENT_NAME").alias(f"unique_events_time_{window_n}_days")
        )
    )
    recency_sub3_agg = recency_sub3_agg.join(window_counts, on="HCP_COHORT_ID", how="left")

# Fill nulls with 0 for count columns
for window_n in time_windows_recency:
    recency_sub3_agg = recency_sub3_agg.fillna(0, 
        subset=[f"total_events_time_{window_n}_days", f"unique_events_time_{window_n}_days"])

print(f"Recency Sub-3 (Counts by Time Window): {recency_sub3_agg.count()} rows")

# --- Sub-feature 4: Counts by Type ---
# Per event type (rx/dx/px/lab): total_{type}, unique_{type}
event_types = [row["EVENT_TYPE"] for row in recency_claims_12m.select("EVENT_TYPE").distinct().collect()]
recency_sub4 = recency_claims_12m.select("HCP_COHORT_ID").distinct()
for etype in event_types:
    type_df = recency_claims_12m.filter(F.col("EVENT_TYPE") == etype)
    type_counts = (type_df
        .groupBy("HCP_COHORT_ID")
        .agg(
            F.count("*").alias(f"total_{etype}"),
            F.countDistinct("EVENT_NAME").alias(f"unique_{etype}")
        )
    )
    recency_sub4 = recency_sub4.join(type_counts, on="HCP_COHORT_ID", how="left")

# Fill nulls with 0
for etype in event_types:
    recency_sub4 = recency_sub4.fillna(0, subset=[f"total_{etype}", f"unique_{etype}"])

print(f"Recency Sub-4 (Counts by Type): {recency_sub4.count()} rows")

# --- Sub-feature 5: Counts by Type x Window ---
# Cross of types x windows: total_events_time_{N}_days_{type}, unique_events_time_{N}_days_{type}
recency_sub5 = recency_claims_12m.select("HCP_COHORT_ID").distinct()
for etype in event_types:
    for window_n in time_windows_recency:
        tw_df = recency_claims_12m.filter(
            (F.col("EVENT_TYPE") == etype) & (F.col("EVENT_TIME") <= window_n)
        )
        tw_counts = (tw_df
            .groupBy("HCP_COHORT_ID")
            .agg(
                F.count("*").alias(f"total_events_time_{window_n}_days_{etype}"),
                F.countDistinct("EVENT_NAME").alias(f"unique_events_time_{window_n}_days_{etype}")
            )
        )
        recency_sub5 = recency_sub5.join(tw_counts, on="HCP_COHORT_ID", how="left")

# Fill nulls with 0
for etype in event_types:
    for window_n in time_windows_recency:
        recency_sub5 = recency_sub5.fillna(0, 
            subset=[f"total_events_time_{window_n}_days_{etype}", 
                    f"unique_events_time_{window_n}_days_{etype}"])

print(f"Recency Sub-5 (Counts by Type x Window): {recency_sub5.count()} rows")

# --- Sub-feature 6: Skewness ---
# skewness(EVENT_TIME) per type x window
# Returns NULL when fewer than 3 events exist or zero variance
recency_sub6 = recency_claims_12m.select("HCP_COHORT_ID").distinct()
for etype in event_types:
    for window_n in time_windows_recency:
        tw_df = recency_claims_12m.filter(
            (F.col("EVENT_TYPE") == etype) & (F.col("EVENT_TIME") <= window_n)
        )
        skew_val = (tw_df
            .groupBy("HCP_COHORT_ID")
            .agg(F.skewness("EVENT_TIME").alias(f"skewness_time_{window_n}_days_{etype}"))
        )
        recency_sub6 = recency_sub6.join(skew_val, on="HCP_COHORT_ID", how="left")

print(f"Recency Sub-6 (Skewness): {recency_sub6.count()} rows")

In [0]:
# ============================================================================
# RECENCY - Combine all sub-features (Section 4)
# ============================================================================
# All sub-features join on HCP_COHORT_ID

recency_combined = recency_sub1_result \
    .join(recency_sub2, on="HCP_COHORT_ID", how="left") \
    .join(recency_sub3_agg, on="HCP_COHORT_ID", how="left") \
    .join(recency_sub4, on="HCP_COHORT_ID", how="left") \
    .join(recency_sub5, on="HCP_COHORT_ID", how="left") \
    .join(recency_sub6, on="HCP_COHORT_ID", how="left")

# Save output (Section 7.7: partition by COHORT)
# Extract COHORT from HCP_COHORT_ID for partitioning
recency_combined = recency_combined.withColumn(
    "COHORT", F.substring(F.col("HCP_COHORT_ID"), -3, 3)
)

recency_output_path = f"{output_base_path}/rx_recency"
recency_combined.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .partitionBy("COHORT") \
    .save(recency_output_path)

print(f"Recency combined: {recency_combined.count()} rows, {len(recency_combined.columns)} columns")
display(recency_combined.limit(10))

---
# Feature Type 3: Frequency (Section 5)

## What it answers
*How often does each event occur at the HCP level, and is the rate accelerating or decelerating?*

Four sub-types, each saved as a separate Delta/Parquet output.

| Sub-type | What it measures | Grain |
|---|---|---|
| A | Monthly claim count per event | `[PROVIDER_ID, SVC_DT_M]` |
| B | Rolling cumulative counts (1,2,3,6,12 months) | `[PROVIDER_ID, SVC_DT_M]` |
| C | Change in frequency vs 12M baseline | `[PROVIDER_ID, SVC_DT_M]` |
| D | Unique patient counts per HCP per event per window | `[PROVIDER_ID, COHORT]` |

## RX-specific column naming for Sub-D
`NUM_OF_PATIENTS_THE_HCP_PRESCRIBED_WITH_{EVENT}_IN_LAST_{N}_DAYS`

In [0]:
# ============================================================================
# FREQUENCY - Sub-type A: Monthly Claim Count (Section 5)
# ============================================================================
# Raw count of each event per HCP per month
# Input scope: All standardized claims (no cohort or lookback filter)
# Grain: [PROVIDER_ID, SVC_DT_M]

# Assign each claim to its monthly bucket
freq_claims = rx_claims \
    .filter(F.col("PROVIDER_ID").isNotNull()) \
    .filter(F.col("EVENT_NAME").isNotNull()) \
    .withColumn("SVC_DT_M", F.date_trunc("MM", F.col("EVENT_DATE")))

# Pivot: groupBy(PROVIDER_ID, SVC_DT_M).pivot(EVENT_NAME).agg(count(*))
freq_subA = (freq_claims
    .groupBy("PROVIDER_ID", "SVC_DT_M")
    .pivot("EVENT_NAME")
    .agg(F.count(F.lit(1)))
)

# Sanitize column names: prefix FREQ_, apply sanitization rules
id_cols_a = ["PROVIDER_ID", "SVC_DT_M"]
rename_exprs_a = []
for col_name in freq_subA.columns:
    if col_name not in id_cols_a:
        sanitized = sanitize_column_name(col_name)
        rename_exprs_a.append(F.col(f"`{col_name}`").alias(f"FREQ_{sanitized}"))
    else:
        rename_exprs_a.append(F.col(col_name))
freq_subA = freq_subA.select(*rename_exprs_a)

# Save Sub-type A output
freq_subA_path = f"{output_base_path}/rx_freq_subA_monthly_count"
freq_subA.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .partitionBy("SVC_DT_M") \
    .save(freq_subA_path)

print(f"Freq Sub-A (Monthly Claim Count): {freq_subA.count()} rows, {len(freq_subA.columns)} columns")
display(freq_subA.limit(10))

In [0]:
# ============================================================================
# FREQUENCY - Sub-type B: Lag Frequency / Rolling Windows (Section 5)
# ============================================================================
# Rolling cumulative event counts over trailing windows of 1, 2, 3, 6, and 12 months
# Three-step process: fill_missing -> lag -> process_and_join

# --- Step 1: fill_missing ---
# Create dense spine: cross-join all distinct PROVIDER_IDs with all months from min to max date
def fill_missing(df, group_col="PROVIDER_ID", date_col="SVC_DT_M"):
    """Create dense spine by cross-joining all HCPs with all months."""
    # Get all distinct HCPs
    all_hcps = df.select(group_col).distinct()
    
    # Get all months from min to max date
    min_date = df.select(F.min(date_col)).collect()[0][0]
    max_date = df.select(F.max(date_col)).collect()[0][0]
    
    # Generate month sequence
    month_seq = spark.createDataFrame(
        pd.date_range(start=min_date, end=max_date, freq='MS'), 
        schema=StructType([StructField(date_col, DateType(), True)])
    )
    
    # Cross-join to create dense spine
    spine = all_hcps.crossJoin(month_seq)
    
    # Left join original data, fillna(0)
    result = spine.join(df, on=[group_col, date_col], how="left")
    
    # Fill nulls with 0 for all non-key columns
    fill_cols = [c for c in result.columns if c not in [group_col, date_col]]
    result = result.fillna(0, subset=fill_cols)
    
    return result

# --- Step 2: lag ---
# Rolling window sums via Window.partitionBy(PROVIDER_ID).orderBy(SVC_DT_M)
def apply_lag(df, windows, group_col="PROVIDER_ID", date_col="SVC_DT_M"):
    """Apply rolling window sums for each window size -- batch .withColumns() to avoid deep plan."""
    w = Window.partitionBy(group_col).orderBy(date_col)
    
    freq_cols = [c for c in df.columns if c not in [group_col, date_col]]
    new_cols = {}
    for col_name in freq_cols:
        for n_months in windows:
            new_cols[f"{col_name}_IN_LAST_{n_months}_MONTH"] = (
                F.sum(col_name).over(w.rowsBetween(-n_months + 1, 0))
            )
    
    return df.withColumns(new_cols)

# --- Step 3: process_and_join ---
# Orchestrator: fill_missing -> lag -> drop raw columns, keep only _IN_LAST_ columns + keys
def process_and_join(df, windows):
    """Full pipeline: fill_missing -> lag -> drop raw freq columns."""
    # Step 1: fill missing
    filled = fill_missing(df)
    
    # Step 2: apply lag
    lagged = apply_lag(filled, windows)
    
    # Step 3: drop raw FREQ_ columns, keep only _IN_LAST_ columns + keys
    key_cols = ["PROVIDER_ID", "SVC_DT_M"]
    keep_cols = key_cols + [c for c in lagged.columns if "_IN_LAST_" in c]
    result = lagged.select(*keep_cols)
    
    return result

# Execute Sub-type B pipeline
freq_subB = process_and_join(freq_subA, lag_windows_freq)

# Save Sub-type B output
freq_subB_path = f"{output_base_path}/rx_freq_subB_lag"
freq_subB.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .partitionBy("SVC_DT_M") \
    .save(freq_subB_path)

print(f"Freq Sub-B (Lag Frequency): {freq_subB.count()} rows, {len(freq_subB.columns)} columns")
display(freq_subB.limit(10))

In [0]:
# ============================================================================
# FREQUENCY - Sub-type C: Change in Frequency (Section 5)
# ============================================================================
# Whether the HCP's event rate is accelerating or decelerating vs its own 12M baseline
# Positive = recent activity exceeds remainder rate; Negative = slowed down
#
# Formula:
# CHANGE = (FREQ_{EVENT}_IN_LAST_{N}_MONTH / N)
#        - ((FREQ_{EVENT}_IN_LAST_12_MONTH - FREQ_{EVENT}_IN_LAST_{N}_MONTH) / (12 - N))
#
# Recent windows: [1, 2, 3, 6] (12M is the baseline, not computed)
# After computing, raw FREQ_*_IN_LAST_* columns are dropped.

# Get all FREQ columns that have _IN_LAST_ suffix
freq_lag_cols = [c for c in freq_subB.columns if "_IN_LAST_" in c and c not in ["PROVIDER_ID", "SVC_DT_M"]]

# Group columns by event (strip the _IN_LAST_{N}_MONTH suffix to get event name)
event_names_in_freq = set()
for col_name in freq_lag_cols:
    # Parse: FREQ_{EVENT}_IN_LAST_{N}_MONTH
    parts = col_name.split("_IN_LAST_")
    event_base = parts[0]  # FREQ_{EVENT}
    event_names_in_freq.add(event_base)

# Compute change for each event and each recent window -- batch .withColumns() to avoid deep plan
new_cols = {}
for event_base in event_names_in_freq:
    # Get the 12M baseline column
    col_12m = f"{event_base}_IN_LAST_12_MONTH"
    if col_12m not in freq_subB.columns:
        continue
    
    for n_months in change_windows_freq:
        col_n = f"{event_base}_IN_LAST_{n_months}_MONTH"
        if col_n not in freq_subB.columns:
            continue
        
        # Compute change
        # CHANGE = (FREQ_N / N) - ((FREQ_12 - FREQ_N) / (12 - N))
        change_col_name = f"CHANGE_IN_FREQ_OF_HCP_CLAIMS_FOR_{event_base.replace('FREQ_', '')}_IN_LAST_{n_months}_MONTH"
        new_cols[change_col_name] = (
            (F.col(col_n) / F.lit(n_months)) -
            ((F.col(col_12m) - F.col(col_n)) / F.lit(12 - n_months))
        )

freq_subC = freq_subB.withColumns(new_cols)

# Select only CHANGE_IN_FREQ columns + keys
key_cols_c = ["PROVIDER_ID", "SVC_DT_M"]
change_cols = [c for c in freq_subC.columns if c.startswith("CHANGE_IN_FREQ_")]
freq_subC = freq_subC.select(*key_cols_c, *change_cols)

# Prerequisite: HCP universe cross-join with cohort anchor dates
# (For joining monthly-grain features to cohort-grain downstream)
# hcp_universe = hcp_universe.crossJoin(cohort_anchor_dates)
# lag_freq = hcp_universe.join(freq_subC, on=["PROVIDER_ID", "SVC_DT_M"], how="inner")

# Save Sub-type C output
freq_subC_path = f"{output_base_path}/rx_freq_subC_change"
freq_subC.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .partitionBy("SVC_DT_M") \
    .save(freq_subC_path)

print(f"Freq Sub-C (Change in Frequency): {freq_subC.count()} rows, {len(freq_subC.columns)} columns")
display(freq_subC.limit(10))

In [0]:
# ============================================================================
# FREQUENCY - Sub-type D: Unique Patient Counts per HCP (Section 5)
# ============================================================================
# Number of distinct patients an HCP has seen for each event type within time windows
# from the cohort's lookback end date.
#
# Grain: [PROVIDER_ID, COHORT] (NOT monthly)
# Time windows: [30, 60, 90, 180, 360] days
# RX column naming: NUM_OF_PATIENTS_THE_HCP_PRESCRIBED_WITH_{EVENT}_IN_LAST_{N}_DAYS

# Step 1: Cross-join claims with broadcast cohort anchor dates
freq_subD_claims = (rx_claims
    .filter(F.col("PROVIDER_ID").isNotNull())
    .filter(F.col("EVENT_NAME").isNotNull())
    .filter(F.col("PATIENT_ID").isNotNull())
    .crossJoin(F.broadcast(cohort_anchor_dates))
)

# Step 2: Filter to lookback window
freq_subD_claims = freq_subD_claims.filter(
    (F.col("EVENT_DATE") >= F.col("LOOKBACK_START_DATE")) &
    (F.col("EVENT_DATE") <= F.col("LOOKBACK_END_DATE"))
)

# Step 3: Compute days_diff = datediff(LOOKBACK_END_DATE, EVENT_DATE)
freq_subD_claims = freq_subD_claims.withColumn(
    "days_diff", F.datediff(F.col("LOOKBACK_END_DATE"), F.col("EVENT_DATE"))
)

# Step 4: Flag: when(days_diff <= N, 1).otherwise(0) for each window
for window_n in patient_count_windows_freq:
    freq_subD_claims = freq_subD_claims.withColumn(
        f"within_{window_n}_days",
        F.when(F.col("days_diff") <= window_n, F.lit(1)).otherwise(F.lit(0))
    )

# Step 5: Dedup on [PROVIDER_ID, COHORT, EVENT_NAME, PATIENT_ID]
dedup_cols = ["PROVIDER_ID", "COHORT", "EVENT_NAME", "PATIENT_ID"]
freq_subD_claims = freq_subD_claims.dropDuplicates(dedup_cols)

# Step 6 & 7: countDistinct(PATIENT_ID) per [PROVIDER_ID, COHORT, EVENT_NAME] per flag
# Then unpivot -> pivot wide with descriptive column names
freq_subD_results = []

for window_n in patient_count_windows_freq:
    # Filter to patients within this window
    window_df = freq_subD_claims.filter(F.col(f"within_{window_n}_days") == 1)
    
    # Count distinct patients per HCP per cohort per event
    counts = (window_df
        .groupBy("PROVIDER_ID", "COHORT", "EVENT_NAME")
        .agg(F.countDistinct("PATIENT_ID").alias("patient_count"))
    )
    
    # Pivot wide by EVENT_NAME with RX-specific naming
    pivoted = (counts
        .groupBy("PROVIDER_ID", "COHORT")
        .pivot("EVENT_NAME")
        .agg(F.sum("patient_count"))
    )
    
    # Rename columns to RX-specific naming convention
    id_cols_d = ["PROVIDER_ID", "COHORT"]
    rename_exprs_d = []
    for col_name in pivoted.columns:
        if col_name not in id_cols_d:
            new_name = f"NUM_OF_PATIENTS_THE_HCP_PRESCRIBED_WITH_{sanitize_column_name(col_name)}_IN_LAST_{window_n}_DAYS"
            rename_exprs_d.append(F.col(f"`{col_name}`").alias(new_name))
        else:
            rename_exprs_d.append(F.col(col_name))
    pivoted = pivoted.select(*rename_exprs_d)
    
    freq_subD_results.append(pivoted)

# Join all window results on [PROVIDER_ID, COHORT]
freq_subD = freq_subD_results[0]
for result in freq_subD_results[1:]:
    freq_subD = freq_subD.join(result, on=["PROVIDER_ID", "COHORT"], how="left")

# Save Sub-type D output
freq_subD_path = f"{output_base_path}/rx_freq_subD_patient_counts"
freq_subD.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .partitionBy("COHORT") \
    .save(freq_subD_path)

print(f"Freq Sub-D (Unique Patient Counts): {freq_subD.count()} rows, {len(freq_subD.columns)} columns")
display(freq_subD.limit(10))

---
# Feature Type 4: Hospitalization (Section 6)

## NOT APPLICABLE FOR RX

Per the skill specification (Section 6):
> **DX and PX only** — no RX or Lab hospitalization features exist.

Hospitalization features require `DATA_SOURCE = 'HX'` (hospitalized claims) and are only computed for Diagnosis (DX) and Procedure (PX) claim types. RX claims do not have a hospitalization dimension.

The hospitalization notebook sections are implemented in the DX and PX notebooks.

In [0]:
# ============================================================================
# FINAL OUTPUT SUMMARY AND JOIN STRATEGY (Section 8)
# ============================================================================

# Per Section 8, all feature outputs share a common key for unified feature table:
#
# | Feature Type            | Key                        | Grain           |
# |---|---|---|
# | First Occurrence        | HCP_COHORT_ID              | HCP x Cohort    |
# | Recency                 | HCP_COHORT_ID              | HCP x Cohort    |
# | Frequency Sub-A,B,C    | PROVIDER_ID, SVC_DT_M      | HCP x Month     |
# | Frequency Sub-D         | PROVIDER_ID, COHORT        | HCP x Cohort    |
#
# Unification key: HCP_COHORT_ID = {PROVIDER_ID}_{COHORT}
# Monthly-grain features (Freq A-C) must be joined to cohort anchor dates to
# convert SVC_DT_M -> COHORT -> HCP_COHORT_ID before merging.

print("=" * 70)
print("RX PATIENT FEATURES - GENERATION COMPLETE")
print("=" * 70)
print()
print("Feature outputs generated:")
print()
print("1. First Occurrence (HCP_COHORT_ID grain)")
print(f"   Path: {output_base_path}/rx_first_occurrence")
print(f"   Columns: max(EVENT_TIME)_{{EVENT_NAME}}_first_occurrence")
print()
print("2. Recency (HCP_COHORT_ID grain, 6 sub-features)")
print(f"   Path: {output_base_path}/rx_recency")
print(f"   Columns: min(EVENT_TIME)_{{EVENT}}_time_min, total_events, unique_events,")
print(f"            total_events_time_{{N}}_days, unique_events_time_{{N}}_days,")
print(f"            total_{{type}}, unique_{{type}},")
print(f"            total_events_time_{{N}}_days_{{type}}, unique_events_time_{{N}}_days_{{type}},")
print(f"            skewness_time_{{N}}_days_{{type}}")
print()
print("3. Frequency Sub-A: Monthly Claim Count (PROVIDER_ID x SVC_DT_M)")
print(f"   Path: {output_base_path}/rx_freq_subA_monthly_count")
print(f"   Columns: FREQ_{{EVENT}}")
print()
print("3. Frequency Sub-B: Lag Frequency (PROVIDER_ID x SVC_DT_M)")
print(f"   Path: {output_base_path}/rx_freq_subB_lag")
print(f"   Columns: FREQ_{{EVENT}}_IN_LAST_{{N}}_MONTH (N = {lag_windows_freq})")
print()
print("3. Frequency Sub-C: Change in Frequency (PROVIDER_ID x SVC_DT_M)")
print(f"   Path: {output_base_path}/rx_freq_subC_change")
print(f"   Columns: CHANGE_IN_FREQ_OF_HCP_CLAIMS_FOR_{{EVENT}}_IN_LAST_{{N}}_MONTH (N = {change_windows_freq})")
print()
print("3. Frequency Sub-D: Unique Patient Counts (PROVIDER_ID x COHORT)")
print(f"   Path: {output_base_path}/rx_freq_subD_patient_counts")
print(f"   Columns: NUM_OF_PATIENTS_THE_HCP_PRESCRIBED_WITH_{{EVENT}}_IN_LAST_{{N}}_DAYS (N = {patient_count_windows_freq})")
print()
print("4. Hospitalization: NOT APPLICABLE FOR RX")
print()
print("=" * 70)
print("Join strategy for unified feature table (Section 8.2):")
print("  Step 1: Build skeleton from hcp_universe x cohort_anchor_dates")
print("  Step 2: LEFT JOIN first_occurrence ON HCP_COHORT_ID")
print("  Step 3: LEFT JOIN recency ON HCP_COHORT_ID")
print("  Step 4: LEFT JOIN freq_subABC ON [PROVIDER_ID, SVC_DT_M] (convert to HCP_COHORT_ID)")
print("  Step 5: LEFT JOIN freq_subD ON [PROVIDER_ID, COHORT]")
print("  Step 6: RX features: no suffix (baseline)")
print("=" * 70)